# 05 - Crown architecture in the two cultivation systems

The results generated here were used in manuscript section 3.7. At the end of
the 25-month observation period, before
pruning, every tree was scored for its primary branches: how many, how long
(short < 0.75 m, medium 0.75-1.5 m, long > 1.5 m) and how they were inserted
(erect or plagiotropic).

The raw sheets use the field shorthand `sol` (sun = monoculture) and `sombra`
(shade = agroforestry). Each branch carries a two-letter `descricao` code:
the first letter is the length class (c/m/l) and the second the orientation
(e/p).

**Produces:** Figure 7.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu

from yerbamate import config as C
from yerbamate import plotting as P, stats as S

P.use_paper_style()
pd.set_option("display.width", 170)


def read_sheet(path, system):
    d = pd.read_excel(path, sheet_name=0, header=4)
    d.columns = ["planta", "galho", "sexo", "descr", "short", "medium", "long", "total"]
    d["planta"] = d["planta"].ffill()
    d["sexo"] = d["sexo"].ffill()
    d["sys"] = system
    return d


arch = pd.concat([read_sheet(C.RAW_ARCH_SUN, "MO"),
                  read_sheet(C.RAW_ARCH_SHADE, "AFS")], ignore_index=True)
arch["length_class"] = arch["descr"].map(
    lambda x: {"c": "short", "m": "medium", "l": "long"}.get(str(x)[0], np.nan)
    if pd.notna(x) else np.nan)
arch["orientation"] = arch["descr"].map(
    lambda x: {"e": "erect", "p": "plagiotropic"}.get(str(x)[1] if len(str(x)) > 1 else "", np.nan)
    if pd.notna(x) else np.nan)

# One row per plant carries the branch totals; three rows per plant describe the
# tagged principal branches.
per_plant = arch.dropna(subset=["total"]).copy()
for c in ["short", "medium", "long"]:
    per_plant[c] = per_plant[c].fillna(0)
per_branch = arch.dropna(subset=["descr"]).copy()

print(f"{len(per_plant)} plants ({per_plant.groupby('sys').size().to_dict()}), "
      f"{len(per_branch)} described branches")

## Branch number

Monoculture plants carried about five times as many primary branches as
agroforestry plants. Within monoculture males branched more than females, but
with only 10 and 5 plants per sex this is reported as a trend, not an effect.
In the shade the sexes branched alike.

In [ ]:
mo = per_plant[per_plant.sys == "MO"]["total"]
afs = per_plant[per_plant.sys == "AFS"]["total"]
U_sys, p_sys = mannwhitneyu(mo, afs, alternative="two-sided")
print(f"MO {mo.mean():.1f} +/- {mo.std():.1f} (n={len(mo)})   "
      f"AFS {afs.mean():.1f} +/- {afs.std():.1f} (n={len(afs)})")
print(f"Mann-Whitney U = {U_sys:.0f}, p = {p_sys:.2e} {S.stars(p_sys)}")

rows = []
for system in ["MO", "AFS"]:
    f = per_plant[(per_plant.sys == system) & (per_plant.sexo == "F")]["total"]
    m = per_plant[(per_plant.sys == system) & (per_plant.sexo == "M")]["total"]
    _, p = mannwhitneyu(f, m, alternative="two-sided")
    rows.append({"System": system, "mean_FE": round(f.mean(), 1), "n_FE": len(f),
                 "mean_MA": round(m.mean(), 1), "n_MA": len(m),
                 "p": round(p, 4), "sig": S.stars(p)})
by_sex = pd.DataFrame(rows)
by_sex.to_csv(C.TAB_DIR / "Figure_7_branching_by_sex.csv", index=False)
print()
print(by_sex.to_string(index=False))

srh = S.scheirer_ray_hare(per_plant, "total", "sexo", "sys")
print("\nScheirer-Ray-Hare on branch number:")
for term, out in srh.items():
    print(f"  {term:14s} H = {out['H']:6.2f}  p = {out['p']:.3f}  {out['sig']}")

## Length composition and orientation

Monoculture crowns are dominated by medium branches - dense and compact. In
agroforestry most of the few branches fall in the long class and almost all are
erect: plants grow upward to escape the shade. With light abundant overhead,
monoculture plants instead spread laterally.

The length data exist at two units; results from both were used in the manuscript:

- **every branch on every tree**, tallied per plant - these are the percentages
  plotted in Figure 7B;
- **the three tagged principal branches per tree** (45 per system) - the n = 45
  the figure caption cites, and the unit of the chi-square quoted in the text.

Both point the same way. The tagged-branch test is the conservative one,
because it does not treat many branches on one tree as independent
observations, so it is the p-value annotated on the figure.

In [ ]:
length_counts = per_plant.groupby("sys")[["short", "medium", "long"]].sum()
length_pct = length_counts.div(length_counts.sum(axis=1), axis=0) * 100
chi2_all, p_len_all, _, _ = chi2_contingency(length_counts.values)
print(f"all primary branches, tallied per plant (n = {int(length_counts.values.sum())}):")
print(length_counts.assign(**{f"{c}_pct": length_pct[c].round(1)
                              for c in length_counts.columns}).to_string())
print(f"chi-square = {chi2_all:.2f}, p = {p_len_all:.3e} {S.stars(p_len_all)}")

tagged_length = pd.crosstab(per_branch.sys, per_branch.length_class)
chi2_tagged, p_len, _, _ = chi2_contingency(tagged_length.values)
print(f"\ntagged principal branches (n = {tagged_length.sum(axis=1).iloc[0]} per system):")
print(tagged_length.to_string())
print(f"chi-square = {chi2_tagged:.2f}, p = {p_len:.4f} {S.stars(p_len)}")

orientation = pd.crosstab(per_branch.sys, per_branch.orientation)
_, p_ori = fisher_exact(orientation.values)
orientation_pct = orientation.div(orientation.sum(axis=1), axis=0) * 100
print("\nprimary branch orientation:")
print(orientation.assign(pct_plagiotropic=orientation_pct["plagiotropic"].round(1)).to_string())
print(f"Fisher exact p = {p_ori:.4f} {S.stars(p_ori)}")

length_counts.to_csv(C.TAB_DIR / "Figure_7_length_composition.csv")
tagged_length.to_csv(C.TAB_DIR / "Figure_7_length_tagged_branches.csv")
orientation.to_csv(C.TAB_DIR / "Figure_7_orientation.csv")

## Branching efficiency

Monoculture plants produce ~5x more branches but receive 10-14x more light.
Per unit of daily light received, agroforestry plants are about twice as
efficient at producing branches - an architectural shade-tolerance signal.

In [ ]:
growth = pd.read_csv(C.UNIFIED)
dli = {"MO": growth[growth.environment == "MO"].DLI.mean(),
       "AFS": growth[growth.environment == "FUS"].DLI.mean()}
per_plant["branches_per_mol"] = per_plant.apply(lambda r: r.total / dli[r.sys], axis=1)

eff = pd.DataFrame([
    {"System": s, "branches": round(per_plant[per_plant.sys == s].total.mean(), 1),
     "DLI_mol_m2_d": round(dli[s], 1),
     "branches_per_mol_DLI": round(per_plant[per_plant.sys == s].branches_per_mol.mean(), 2),
     "n": int((per_plant.sys == s).sum())}
    for s in ["MO", "AFS"]])
eff.to_csv(C.TAB_DIR / "Figure_7_branching_efficiency.csv", index=False)
_, p_eff = mannwhitneyu(per_plant[per_plant.sys == "MO"].branches_per_mol,
                        per_plant[per_plant.sys == "AFS"].branches_per_mol,
                        alternative="two-sided")
print(eff.to_string(index=False))
print(f"Mann-Whitney on branches per mol DLI: p = {p_eff:.2e} {S.stars(p_eff)}")

## Figure 7

Following the project convention, panels show p-values rather than test names;
the tests belong in the caption. Panel A is Mann-Whitney U, B chi-square,
C Fisher's exact.

In [ ]:
def fmt(p):
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"


fig, axes = plt.subplot_mosaic("ABC", figsize=(21, 6.8))
groups = [("MO", "F"), ("MO", "M"), ("AFS", "F"), ("AFS", "M")]
positions = [1, 2, 3.5, 4.5]

# --- (A) branch number by system and sex ---
ax = axes["A"]
data = [per_plant[(per_plant.sys == s) & (per_plant.sexo == sx)]["total"].values
        for s, sx in groups]
bp = ax.boxplot(data, positions=positions, widths=0.7, patch_artist=True,
                showfliers=True, medianprops=dict(color="#222", lw=1.9))
for box, (s, sx) in zip(bp["boxes"], groups):
    box.set_facecolor(P.COL_MO if s == "MO" else P.COL_AFS)
    box.set_alpha(0.9)
    box.set_hatch("" if sx == "F" else "//")
for part in ["whiskers", "caps"]:
    for item in bp[part]:
        item.set_color("#666")
for x1, x2, a, b, yy in [(1, 2, data[0], data[1], 80), (3.5, 4.5, data[2], data[3], 22)]:
    ax.plot([x1, x1, x2, x2], [yy, yy + 2, yy + 2, yy], lw=1.2, color="#333")
    ax.text((x1 + x2) / 2, yy + 2.5, fmt(mannwhitneyu(a, b, alternative="two-sided")[1]),
            ha="center", va="bottom", fontsize=15)
ax.plot([1.5, 1.5, 4, 4], [90, 92, 92, 90], lw=1.2, color="#333")
ax.text(2.75, 92.5, fmt(p_sys), ha="center", va="bottom", fontsize=15)
ax.set_xticks(positions)
ax.set_xticklabels(["FE\nMO", "MA\nMO", "FE\nAFS", "MA\nAFS"], fontsize=17)
ax.set_ylabel("Primary branches / plant", fontsize=19)
ax.set_ylim(0, 100)
P.despine(ax)
ax.legend(handles=[mpatches.Patch(facecolor="#cccccc", edgecolor="#888", label="Female (FE)"),
                   mpatches.Patch(facecolor="#cccccc", edgecolor="#888", hatch="//",
                                  label="Male (MA)")],
          loc="upper right", frameon=False, fontsize=13, handlelength=1.4)
P.panel(ax, "A", size=24)

# --- (B) length composition ---
ax = axes["B"]
order = ["MO", "AFS"]
bottom = np.zeros(2)
for col, label, colour in [("short", "Short", "#c6dbef"), ("medium", "Medium", "#6baed6"),
                           ("long", "Long", "#08519c")]:
    vals = length_pct.loc[order, col].values
    ax.bar(order, vals, bottom=bottom, color=colour, label=label, edgecolor="white", width=0.6)
    for i, v in enumerate(vals):
        if v > 4:
            ax.text(i, bottom[i] + v / 2, f"{v:.0f}%", ha="center", va="center",
                    color="white", fontsize=16, fontweight="bold")
    bottom += vals
ax.set_ylabel("Primary branches (% by length)", fontsize=19)
ax.set_ylim(0, 100)
ax.text(0.5, 1.01, fmt(p_len), ha="center", fontsize=15, transform=ax.transAxes)
P.despine(ax)
ax.legend(loc="lower center", frameon=False, ncol=3, fontsize=15, bbox_to_anchor=(0.5, -0.26))
P.panel(ax, "B", size=24)

# --- (C) orientation ---
ax = axes["C"]
bottom = np.zeros(2)
for key, label, colour in [("erect", "Erect", "#6a51a3"),
                           ("plagiotropic", "Plagiotropic", "#b8860b")]:
    vals = orientation_pct.loc[order, key].values
    ax.bar(order, vals, bottom=bottom, color=colour, label=label, edgecolor="white", width=0.6)
    for i, v in enumerate(vals):
        if v > 4:
            ax.text(i, bottom[i] + v / 2, f"{v:.0f}%", ha="center", va="center",
                    color="white", fontsize=16, fontweight="bold")
    bottom += vals
ax.set_ylabel("Primary branches (% orientation)", fontsize=19)
ax.set_ylim(0, 100)
ax.text(0.5, 1.01, fmt(p_ori), ha="center", fontsize=15, transform=ax.transAxes)
P.despine(ax)
ax.legend(loc="lower center", frameon=False, ncol=2, fontsize=15, bbox_to_anchor=(0.5, -0.24))
P.panel(ax, "C", size=24)

fig.tight_layout(w_pad=3, rect=[0, 0.04, 1, 1])
P.save(fig, "Figure_7", dpi=500)

In [ ]:
assert abs(mo.mean() - 40.6) < 0.1 and abs(afs.mean() - 8.3) < 0.1
assert p_sys < 0.001
assert abs(p_ori - 0.037) < 0.001
assert abs(p_len - 0.009) < 0.001
assert abs(eff.set_index("System").loc["AFS", "branches_per_mol_DLI"] - 2.91) < 0.01
assert abs(eff.set_index("System").loc["MO", "branches_per_mol_DLI"] - 1.48) < 0.01
print("Validated the Figure 7 architecture results used in the manuscript.")